# Dataset Timelapse Visualization Demo

This notebook demonstrates how to load a `.pt` dataset generated by `data_collection_notebook.ipynb` and display an interactive, scrollable timelapse of a drone swarm across its episode.

**Pre-requisite:** Before using this, you must run the updated data generation procedure that saves `pos` and `obstacles` directly to the data graphs.


In [ ]:
import sys
import os
import torch
import importlib
from pathlib import Path
from torch_geometric.data import InMemoryDataset

# Ensure we can import out visualization tool
sys.path.append(str(Path.cwd().parent))
import visualization.dataset_visualizer

importlib.reload(visualization.dataset_visualizer)
from visualization.dataset_visualizer import visualize_episode_timelapse


# Standard generic loader for generated datasets
class GeneratedSplitDataset(InMemoryDataset):
    def __init__(self, split_path: str | Path):
        self.split_path = Path(split_path)
        payload = torch.load(self.split_path, weights_only=False)
        self.formation_names = payload.get("formation_names", [])
        self.split_name = payload.get("split_name", "unknown")
        super().__init__(root="")
        self.data = payload["data"]
        self.slices = payload["slices"]

### Load the Dataset

Update the dataset name to whatever dataset you most recently generated with `pos` saved.


In [7]:
inspect_dataset_name = "set_point_prediction_dataset_parallel_mixed_formations"
inspect_split_name = "train"

datasets_dir = Path.cwd().parent / "datasets"
split_path = datasets_dir / f"{inspect_dataset_name}_{inspect_split_name}.pt"

if not split_path.exists():
    print(f"Warning: Dataset not found at {split_path}.")
    print("Please run generate_dataset() in the data collection notebook first.")
else:
    dataset = GeneratedSplitDataset(split_path)
    print(f"Loaded {len(dataset)} graphs.")
    # Print unique episodes found
    episodes = set()
    for i in range(min(500, len(dataset))):  # Scanning first few to find an episode ID
        ep_id = dataset[i].episode_id.item() if hasattr(dataset[i], "episode_id") else 0
        episodes.add(ep_id)
    print(f"Found episode IDs (subset preview): {list(episodes)}")

Loaded 13510 graphs.
Found episode IDs (subset preview): [8, 9]


### Visualize an Episode Timelapse

Launch the interactive Plotly viewer. Drag the `Step` slider right and left to scrub through the episode. Turn `view_2d` to `False` if you want a 3D orbital view.


In [11]:
if split_path.exists():
    # Replace 0 with any episode_id found above
    chosen_episode = 0

    fig = visualize_episode_timelapse(
        dataset=dataset,
        episode_id=8,
        title=f"Episode {chosen_episode} Timelapse (X-Y Top-Down View)",
        view_2d=True,  # True for Top-Down (X-Y), False for 3D Orbit
    )

    # fig.show() will render the interactive UI inline in your notebook
    fig.show()